# DQN (Deep Q-Network) — Step by Step

**Environment**: CartPole-v1  
**Algorithm**: Deep Q-Network with Experience Replay + Target Network

---

## Overview

| Component | Role |
|-----------|------|
| Q-Network | Predicts Q(s,a) for each action given a state |
| Target Network | A frozen copy of Q-Network used to compute stable Bellman targets |
| Replay Buffer | Stores past (s,a,r,s') tuples; random sampling breaks correlations |
| ε-greedy | Balances exploration (random) vs exploitation (max Q) |

**Core Update Rule:**
$$y_i = r_i + \gamma \max_{a'} Q_{\text{target}}(s_i', a') \qquad \text{Loss} = \frac{1}{N}\sum_i (Q(s_i, a_i) - y_i)^2$$

## 1. Imports & Dependencies

In [ ]:
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import gymnasium as gym

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)
random.seed(42)

print("Gymnasium:", gym.__version__)
print("PyTorch:  ", torch.__version__)

## 2. Hyperparameters

> **ε decay schedule**: ε starts at 1.0 (pure exploration) and decays by ×0.995 each episode,
> eventually settling at 0.01 (1% random actions). This ensures the agent explores early and
> exploits later once it has learned something useful.

In [ ]:
# ── Training ──────────────────────────────────────
EPISODES           = 500     # Total training episodes
GAMMA              = 0.99    # Discount factor (how much we value future rewards)
ALPHA              = 1e-3    # Learning rate for Adam optimizer

# ── Exploration ────────────────────────────────────
EPSILON_START      = 1.0     # Start: 100% random actions
EPSILON_END        = 0.01    # Floor: at least 1% random (never stop exploring)
EPSILON_DECAY      = 0.995   # Multiply epsilon by this after each episode

# ── Replay Buffer ──────────────────────────────────
BUFFER_SIZE        = 10_000  # Maximum number of stored experiences
BATCH_SIZE         = 64      # How many experiences to sample per training step

# ── Target Network ─────────────────────────────────
TARGET_UPDATE_FREQ = 10      # Copy Q → Q_target every N episodes

print("Hyperparameters loaded.")
print(f"  Epsilon: {EPSILON_START} → {EPSILON_END} (decay: {EPSILON_DECAY})")
print(f"  Buffer: {BUFFER_SIZE:,} | Batch: {BATCH_SIZE}")

## 3. Q-Network Architecture

The neural network maps **(state → Q-values for all actions)**.

```
Input(4)  →  Linear(128) → ReLU  →  Linear(128) → ReLU  →  Linear(2)
                                                               ↑
                                               [Q(s, left), Q(s, right)]
```

**Why linear output activation?**  
Q-values can be any real number (positive or negative). Using `ReLU` or `sigmoid`  
would clamp the output range, preventing the network from expressing negative Q-values.

In [ ]:
class QNetwork(nn.Module):
    """
    Approximates Q(s, a) for all actions simultaneously.
    Input  : state vector  (batch, state_size)
    Output : Q-values      (batch, action_size)
    """
    def __init__(self, state_size, action_size):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_size, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, action_size),   # No activation — Q values are unbounded
        )

    def forward(self, x):
        return self.net(x)


# Quick sanity check
_dummy_net = QNetwork(state_size=4, action_size=2)
_dummy_input = torch.zeros(1, 4)
_dummy_output = _dummy_net(_dummy_input)
print("Input  shape:", _dummy_input.shape)   # (1, 4)
print("Output shape:", _dummy_output.shape)  # (1, 2)  ← one Q-value per action
print("Q-values (random init):", _dummy_output.detach().numpy())

## 4. Replay Buffer

**Why not train on consecutive experiences directly?**

Consecutive steps are highly correlated: step 3 follows step 2, step 4 follows step 3, etc.  
Training on correlated sequences causes the network to overfit to recent transitions  
and "forget" older, valuable experiences.

**The solution**: Store all experiences in a buffer, then *randomly sample* a mini-batch.  
This breaks temporal correlations and gives each experience multiple chances to be learned from.

In [ ]:
class ReplayBuffer:
    """
    Fixed-size queue of (s, a, r, s', done) tuples.
    When full, oldest entries are automatically discarded (deque with maxlen).
    """
    def __init__(self, capacity):
        self.buffer = deque(maxlen=capacity)

    def push(self, s, a, r, s_next, done):
        """Store one transition."""
        self.buffer.append((s, a, r, s_next, done))

    def sample(self, batch_size):
        """
        Randomly draw batch_size experiences.
        Returns 5 tensors ready for PyTorch training.
        """
        batch = random.sample(self.buffer, batch_size)

        # Unzip list of tuples into 5 separate lists, then convert to tensors
        s, a, r, s_next, done = zip(*batch)
        return (
            torch.FloatTensor(np.array(s)),       # (64, 4)
            torch.LongTensor(a),                   # (64,)
            torch.FloatTensor(r),                  # (64,)
            torch.FloatTensor(np.array(s_next)),   # (64, 4)
            torch.FloatTensor(done),               # (64,)  0.0 or 1.0
        )

    def __len__(self):
        return len(self.buffer)


# Demo
_buf = ReplayBuffer(capacity=100)
_buf.push(np.zeros(4), 0, 1.0, np.ones(4), 0.0)
_buf.push(np.ones(4),  1, -1.0, np.zeros(4), 1.0)
print(f"Buffer size: {len(_buf)}")
_s, _a, _r, _sn, _d = _buf.sample(batch_size=2)
print(f"Sampled states shape: {_s.shape}")   # (2, 4)
print(f"Sampled rewards:      {_r}")

## 5. DQN Agent

The agent ties together all components:
- **`act()`** : ε-greedy action selection
- **`remember()`** : store experience in buffer  
- **`train_step()`** : sample batch → compute Bellman targets → gradient descent
- **`update_target()`** : copy Q-network weights to Target network

In [ ]:
class DQNAgent:
    def __init__(self, state_size, action_size):
        self.action_size = action_size
        self.epsilon = EPSILON_START

        # ── Two networks: online (trained) and target (frozen copy) ──
        self.q_network      = QNetwork(state_size, action_size)
        self.target_network = QNetwork(state_size, action_size)
        # Initialise target with same weights as online network
        self.target_network.load_state_dict(self.q_network.state_dict())
        self.target_network.eval()   # Target network: inference only, no gradient

        self.optimizer = optim.Adam(self.q_network.parameters(), lr=ALPHA)
        self.buffer    = ReplayBuffer(BUFFER_SIZE)

    # ────────────────────────────────────────────────────
    def act(self, state):
        """
        ε-greedy action selection.

        With probability ε  → random action (exploration)
        With probability 1-ε → argmax Q(s, a) (exploitation)
        """
        if random.random() < self.epsilon:
            return random.randrange(self.action_size)        # Explore

        state_t  = torch.FloatTensor(state).unsqueeze(0)    # (4,) → (1,4)
        with torch.no_grad():
            q_values = self.q_network(state_t)              # (1, 2)
        return q_values.argmax(dim=1).item()                 # Exploit

    # ────────────────────────────────────────────────────
    def remember(self, s, a, r, s_next, done):
        self.buffer.push(s, a, r, s_next, done)

    # ────────────────────────────────────────────────────
    def train_step(self):
        """
        One gradient update.

        Step 1: Sample a mini-batch from the buffer
        Step 2: Compute Bellman target using the FROZEN target network
                  y = r + γ max_a' Q_target(s', a')    if not done
                  y = r                                  if done
        Step 3: MSE loss between Q_online prediction and y
        Step 4: Backprop and update Q_online weights ONLY
        """
        if len(self.buffer) < BATCH_SIZE:
            return None    # Not enough data yet

        states, actions, rewards, next_states, dones = self.buffer.sample(BATCH_SIZE)

        # ── Step 1: Q_online prediction for the (s, a) pairs we actually took ──
        q_pred = self.q_network(states)                          # (64, 2)
        # .gather: pick the Q-value of the action actually chosen
        q_pred = q_pred.gather(1, actions.unsqueeze(1)).squeeze(1)  # (64,)

        # ── Step 2: Bellman target using frozen Q_target ──────────────────────
        with torch.no_grad():                                    # No gradient through target
            q_next     = self.target_network(next_states)        # (64, 2)
            max_q_next = q_next.max(dim=1).values                # (64,) best next action
            # When done=1, future reward is 0 (episode is over)
            target = rewards + GAMMA * max_q_next * (1 - dones) # (64,)

        # ── Step 3 & 4: Compute loss and update ──────────────────────────────
        loss = nn.MSELoss()(q_pred, target)
        self.optimizer.zero_grad()
        loss.backward()
        nn.utils.clip_grad_norm_(self.q_network.parameters(), max_norm=10)
        self.optimizer.step()
        return loss.item()

    # ────────────────────────────────────────────────────
    def decay_epsilon(self):
        self.epsilon = max(EPSILON_END, self.epsilon * EPSILON_DECAY)

    def update_target(self):
        """Hard update: copy all Q_online weights to Q_target."""
        self.target_network.load_state_dict(self.q_network.state_dict())

print("DQNAgent class defined.")

## 6. Training Loop

Each episode:
1. Reset environment → get initial state
2. Loop until done: act → store → train
3. Decay ε
4. Every `TARGET_UPDATE_FREQ` episodes: sync target network

**When does training trigger?**  
`train_step()` is called **every time step** (not just every episode).  
But it does nothing until the buffer has at least `BATCH_SIZE` entries.

In [ ]:
def train_dqn():
    env   = gym.make("CartPole-v1")
    agent = DQNAgent(
        state_size  = env.observation_space.shape[0],   # 4
        action_size = env.action_space.n,               # 2
    )

    scores       = []
    recent_100   = deque(maxlen=100)

    print("=" * 60)
    print("DQN Training — CartPole-v1")
    print("=" * 60)

    for episode in range(1, EPISODES + 1):
        state, _ = env.reset()
        total_reward = 0

        # ── Inner loop: one episode ───────────────────
        while True:
            action                              = agent.act(state)
            next_state, reward, term, trunc, _ = env.step(action)
            done                               = term or trunc

            agent.remember(state, action, reward, next_state, float(done))
            agent.train_step()    # Train every step

            state        = next_state
            total_reward += reward
            if done:
                break

        # ── Post-episode updates ──────────────────────
        agent.decay_epsilon()
        scores.append(total_reward)
        recent_100.append(total_reward)

        if episode % TARGET_UPDATE_FREQ == 0:
            agent.update_target()

        # ── Logging ──────────────────────────────────
        if episode % 20 == 0:
            avg = np.mean(recent_100)
            print(f"  Ep {episode:4d} | Avg(100): {avg:6.1f} | ε: {agent.epsilon:.3f}")

        # ── Solved? ──────────────────────────────────
        if len(recent_100) == 100 and np.mean(recent_100) >= 475:
            print(f"\n✓ Solved at episode {episode}! Avg(100): {np.mean(recent_100):.1f}")
            break

    env.close()
    return scores, agent

dqn_scores, dqn_agent = train_dqn()

## 7. Results

In [ ]:
import matplotlib.pyplot as plt

def plot_scores(scores, title, window=20, color='steelblue'):
    rolling = [np.mean(scores[max(0,i-window):i+1]) for i in range(len(scores))]
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(scores,  alpha=0.3, color=color, label='Episode score')
    ax.plot(rolling, color=color, linewidth=2, label=f'Rolling avg ({window})')
    ax.axhline(475, color='red', linestyle='--', linewidth=1, label='Solved threshold')
    ax.set_xlabel("Episode")
    ax.set_ylabel("Total reward")
    ax.set_title(title)
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

plot_scores(dqn_scores, "DQN — CartPole-v1 Training Curve", color='steelblue')

print(f"Best episode:       {max(dqn_scores):.0f}")
print(f"Final avg (last 100): {np.mean(dqn_scores[-100:]):.1f}")

## 8. Inspecting Learned Q-values

After training, we can ask: **what does the agent *think* about each state?**

We query the trained Q-network with specific states to see which action it prefers.

In [ ]:
env_eval = gym.make("CartPole-v1")
state, _ = env_eval.reset()

print("Inspecting Q-values for first 5 states:\n")
print(f"{'State (cart_pos, cart_vel, pole_angle, pole_vel)':>50} | Q(left) | Q(right) | Action")
print("-" * 95)

for step in range(5):
    state_t  = torch.FloatTensor(state).unsqueeze(0)
    with torch.no_grad():
        q_vals = dqn_agent.q_network(state_t).squeeze().numpy()

    chosen = np.argmax(q_vals)
    action_label = ["LEFT", "RIGHT"][chosen]
    state_str = f"[{state[0]:+.2f}, {state[1]:+.2f}, {state[2]:+.3f}, {state[3]:+.2f}]"
    print(f"{state_str:>50} | {q_vals[0]:+.3f}  | {q_vals[1]:+.3f}   | {action_label}")

    state, _, done, _, _ = env_eval.step(chosen)
    if done:
        break

env_eval.close()